# Student Performance Prediction — training demonstration

This notebook is a *demonstration layer* over the automated pipeline in `src/`.
The authoritative run is the single command:

```bash
python src/train_model.py
```

which inspects, preprocesses, sanity-checks, trains, validates, tests, saves artifacts
and writes `training_summary.md` + `results/metrics.json`. The notebook below shows the
same stages step by step, and reads the saved artifacts so nothing is retrained
unintentionally.

**Two datasets, two models.** The 1M-record table (study hours / attendance / participation
-> total_score, grade) is the primary model. The 1,000-record exams table (demographics ->
math score) has a different schema and is trained separately.


In [ ]:
import json, os, sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
pd.set_option("display.width", 160)
print("project root:", PROJECT_ROOT)

## 1. Inspect the datasets (read-only)

`src/inspect_data.py` walks the project, classifies every table by schema, and reports
shape, dtypes, missing values, duplicates, outliers, cross-split overlap and the
hardware. It only reads the original parquet files.

In [ ]:
from inspect_data import build_report, render_markdown
report = build_report()
print(render_markdown(report)[:4000])

## 2. Processed splits

Preprocessing is performed by `src/preprocess.py`: duplicate removal inside each split,
type coercion, row-local feature engineering, and a scaler/encoder fitted **on the
training split only**. The target and grade label are never used as inputs.

In [ ]:
from preprocess import load_processed, MODEL_FEATURES, RAW_FEATURES, TARGET_COLUMN
splits = load_processed("million")
{name: df.shape for name, df in splits.items()}

In [ ]:
splits["train"][RAW_FEATURES + [TARGET_COLUMN]].describe().round(2)

In [ ]:
print("features used by the model:", MODEL_FEATURES)
print("target:", TARGET_COLUMN)
print("grade counts (train):")
print(splits["train"]["grade"].value_counts())

## 3. Measured results from the last full run

Numbers below come straight from `results/metrics.json`, produced by actual training on
this machine — no values are typed in by hand.

In [ ]:
metrics = json.loads((PROJECT_ROOT / "results" / "metrics.json").read_text())
print("selected model:", metrics["model"]["name"], metrics["model"]["params"])
print("validation:", {k: round(v, 4) for k, v in metrics["validation_metrics_selected"].items() if isinstance(v, float)})
print("test      :", {k: round(v, 4) for k, v in metrics["test_metrics"].items() if isinstance(v, float)})
comparison = pd.DataFrame(metrics["candidates"])
comparison[["name", "n_train_used", "fit_seconds"]].assign(
    val_mae=comparison.val_metrics.apply(lambda m: round(m["mae"], 3)),
    val_rmse=comparison.val_metrics.apply(lambda m: round(m["rmse"], 3)),
    val_r2=comparison.val_metrics.apply(lambda m: round(m["r2"], 4)))

In [ ]:
print("grade classifier:", metrics["grade_classifier"]["selected"])
print(json.dumps(metrics["grade_classifier"]["test_metrics"], indent=2)[:1200])

In [ ]:
print("exams model test metrics:")
print(json.dumps(metrics["exams_model"]["test_metrics"], indent=2))

## 4. Feature importance

Permutation importance on the held-out test split (increase in RMSE when a feature is
shuffled). Study hours dominate; attendance and participation carry almost no signal.

In [ ]:
importance = pd.Series(metrics["feature_importance_permutation_rmse"]).sort_values(ascending=False)
importance

## 5. Visualisations saved by the pipeline

In [ ]:
from IPython.display import Image, display
for name in ["actual_vs_predicted", "residuals", "feature_importance", "model_comparison",
             "confusion_matrix", "training_curves"]:
    path = PROJECT_ROOT / "results" / "visualizations" / f"{name}.png"
    if path.exists():
        print(name)
        display(Image(filename=str(path), width=520))

## 6. Predictions on the untouched test split

`results/predictions.csv` holds one row per test student with the actual score, the
predicted score, the residual and the predicted grade.

In [ ]:
preds = pd.read_csv(PROJECT_ROOT / "results" / "predictions.csv")
print(preds.shape)
preds.head(10)

In [ ]:
preds[["actual_total_score", "predicted_total_score"]].corr().iloc[0, 1], preds["residual"].abs().mean()

## 7. Natural-language demonstration (not a language model)

`src/nl_demo.py` answers questions from dataset statistics or from the trained tabular
model. Every answer states whether it is an observation or a prediction.

In [ ]:
from nl_demo import StudentPerformanceAssistant
assistant = StudentPerformanceAssistant()
print(assistant.answer("What factors influence student performance?")["answer"])

In [ ]:
print(assistant.answer("Predict the score for a student studying 18 hours with 92% attendance and participation 8")["answer"])

In [ ]:
print(assistant.answer("How does test preparation relate to exam performance?")["answer"])

## 8. Optional: retrain from scratch

Uncomment to run the full automated pipeline from this notebook (a few minutes on CPU).

In [ ]:
# import subprocess, sys
# subprocess.run([sys.executable, "src/train_model.py"], check=True)